In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

# ---- Load all results ----
results_dir = Path("results")

psm_rows = []
mdm_rows = []

for csv_path in results_dir.glob("PSM_balance_*.csv"):
    site_id = csv_path.stem.replace("PSM_balance_", "")
    df = pd.read_csv(csv_path).dropna(subset=["covariate"])
    df["site_id"] = site_id
    df["method"] = "PSM"
    psm_rows.append(df)

for csv_path in results_dir.glob("MDM_balance_*.csv"):
    site_id = csv_path.stem.replace("MDM_balance_", "")
    df = pd.read_csv(csv_path).dropna(subset=["covariate"])
    df["site_id"] = site_id
    df["method"] = "MDM"
    mdm_rows.append(df)

psm_df = pd.concat(psm_rows, ignore_index=True)
mdm_df = pd.concat(mdm_rows, ignore_index=True)
all_df = pd.concat([psm_df, mdm_df], ignore_index=True)

print(f"PAs in PSM results: {psm_df['site_id'].nunique()}")
print(f"PAs in MDM results: {mdm_df['site_id'].nunique()}")
common_pas = set(psm_df['site_id']) & set(mdm_df['site_id'])
print(f"PAs with both methods: {len(common_pas)}")

# Restrict to PAs with both methods for fair comparison
all_df = all_df[all_df["site_id"].isin(common_pas)]

In [ ]:
# ==== 1. Aggregate comparison: overall mean balance ====
print("\n=== Aggregate comparison ===")
agg = all_df.groupby("method").agg({
    "mean_abs_smd": ["mean", "median"],
    "p90_abs_smd": ["mean", "median"],
}).round(3)
print(agg)

In [ ]:
# ==== 2. Per-PA comparison: head-to-head ====
print("\n=== Per-PA comparison ===")
# Compute mean of mean_abs_smd across covariates for each PA × method
per_pa = (
    all_df
    .groupby(["site_id", "method"])["mean_abs_smd"]
    .mean()
    .unstack()  # columns: MDM, PSM
)
per_pa["mdm_better_by"] = per_pa["PSM"] - per_pa["MDM"]
per_pa["winner"] = np.where(per_pa["MDM"] < per_pa["PSM"], "MDM", "PSM")
per_pa = per_pa.sort_values("mdm_better_by", ascending=False)

print(per_pa)
print(f"\nMDM wins: {(per_pa['winner'] == 'MDM').sum()} / {len(per_pa)} PAs")
print(f"PSM wins: {(per_pa['winner'] == 'PSM').sum()} / {len(per_pa)} PAs")
print(f"Mean improvement (PSM - MDM): {per_pa['mdm_better_by'].mean():+.4f}")
print(f"Median improvement: {per_pa['mdm_better_by'].median():+.4f}")

In [ ]:
# ==== 3. Per-covariate comparison ====
print("\n=== Per-covariate comparison ===")
per_cov = (
    all_df
    .groupby(["covariate", "method"])
    .agg({"mean_abs_smd": "mean", "p90_abs_smd": "mean"})
    .unstack()
)
print(per_cov.round(3))

In [ ]:
# ==== 4. Heterogeneity check: p90 vs mean ratio ====
print("\n=== Heterogeneity check: ratio of p90 to mean ===")
all_df["heterogeneity_ratio"] = all_df["p90_abs_smd"] / all_df["mean_abs_smd"].replace(0, np.nan)
het = all_df.groupby("method")["heterogeneity_ratio"].agg(["mean", "median"]).round(2)
print(het)
print("Higher ratios indicate that worst matches are much worse than typical matches.")

In [ ]:
# ==== 5. Outlier identification: PAs with worst balance per method ====
print("\n=== Worst-balanced PAs under each method ===")
for method in ["PSM", "MDM"]:
    method_df = all_df[all_df["method"] == method]
    worst = (
        method_df.groupby("site_id")["mean_abs_smd"]
        .mean()
        .sort_values(ascending=False)
        .head(5)
    )
    print(f"\nWorst {method} PAs:")
    print(worst.round(3))

# Are the same PAs worst under both methods?
psm_worst_5 = set(
    all_df[all_df["method"] == "PSM"].groupby("site_id")["mean_abs_smd"].mean().sort_values(ascending=False).head(5).index
)
mdm_worst_5 = set(
    all_df[all_df["method"] == "MDM"].groupby("site_id")["mean_abs_smd"].mean().sort_values(ascending=False).head(5).index
)
overlap = psm_worst_5 & mdm_worst_5
print(f"\nPAs in worst-5 under both methods: {sorted(overlap)}")
print("(These PAs may have structural data issues, not method issues)")

In [ ]:
# Bonus: count how many covariates each method got into "acceptable" balance per PA
print("\n=== Covariates in acceptable balance (mean_abs_smd < 0.25) ===")
all_df["is_acceptable"] = all_df["mean_abs_smd"] < 0.25
balance_count = (
    all_df.groupby(["site_id", "method"])["is_acceptable"]
    .sum()
    .unstack()
)
balance_count["mdm_advantage"] = balance_count["MDM"] - balance_count["PSM"]
print(balance_count.sort_values("mdm_advantage", ascending=False))